# Skripsi — Training Deteksi Biji Kakao dari Awal

**Deteksi dan Klasifikasi Tingkat Fermentasi Biji Kakao Menggunakan
SSDLite-MobileNetV3-Large Berbasis Analisis Citra Bentuk dan Warna**

Notebook ini adalah implementasi penelitian mandiri. Semua kode utama berada
langsung di dalam sel notebook: membaca COCO Roboflow, audit data, preprocessing,
augmentasi, model, training, evaluasi, dan prediksi foto baru.
**Tidak memerlukan `src/`, `config.py`, `run_pipeline.py`, atau website.**

“Dari awal” berarti memulai **eksperimen kakao baru dari dataset ekspor Roboflow**,
tanpa checkpoint kakao lama. Sesuai metode skripsi, bobot awal tetap memakai
pretrained detection COCO TorchVision (**transfer learning**), lalu dilatih pada
tiga kelas kakao. Ini berbeda dari melatih seluruh bobot secara acak.

| Label model | Kelas |
|---|---|
| 0 | background — bukan biji yang dihitung |
| 1 | Tidak Fermentasi |
| 2 | Setengah Fermentasi |
| 3 | Full Fermentasi |

Satu foto dapat berisi banyak biji dan beberapa kelas sekaligus. Model mempelajari
pola bentuk dan warna dari RGB; baseline ini tidak menghitung fitur warna/bentuk
secara manual. Hasil penelitian baru tersedia setelah dataset Anda dilatih.

## Sebelum menekan ▶

1. Buat satu folder baru, misalnya `SKRIPSI_KAKAO_NOTEBOOK`.
2. Simpan **file notebook ini** di dalamnya. Ekstrak ZIP COCO Roboflow ke folder
   `dataset` seperti struktur berikut. Gambar dan JSON tetap berada dalam splitnya.
3. Buka folder tersebut di VS Code. Pasang ekstensi **Python** dan **Jupyter** dari
   Microsoft jika belum ada. Klik **Select Kernel → Python Environments** dan pilih
   Python 3.12 / environment GPU yang sebelumnya dipakai training. Environment
   boleh berada di proyek lama; kode proyek lama tidak diperlukan.
4. Jalankan sel kode dari atas ke bawah dengan tombol ▶. Tidak perlu mengetik
   perintah terminal. Baca penjelasan sebelum setiap tahap.

```text
SKRIPSI_KAKAO_NOTEBOOK/
├── KAKAO_TRAINING_DARI_NOL.ipynb
├── dataset/
│   ├── train/   # _annotations.coco.json dan gambar train
│   ├── valid/   # _annotations.coco.json dan gambar validation; val juga boleh
│   └── test/    # _annotations.coco.json dan gambar test
└── hasil_penelitian/  # dibuat otomatis; setiap eksperimen memiliki folder sendiri
```

**Run All menjalankan training** sesuai `NUM_EPOCHS` (awal 100), lalu memilih
threshold pada validation. Evaluasi test hanya dijalankan setelah pilihan final
diaktifkan pada sel test. Tombol **Interrupt** menghentikan sel yang sedang berjalan;
checkpoint epoch terakhir yang selesai tersimpan untuk resume.

Seed mengurangi variasi; angka lintas perangkat/versi dan sesudah resume dapat
memiliki perbedaan numerik kecil. Jangan menafsirkan seed sebagai jaminan hasil identik.

Dataset sudah berlabel COCO; tidak perlu melabel ulang. Jika belum dibagi, lakukan
pembagian berdasarkan sesi pengambilan sebelum ekspor. Hindari foto sangat mirip
dari sesi yang sama tersebar ke train/validation/test. Augmentasi file di Roboflow
tidak diperlukan untuk baseline ini karena augmentasi train dilakukan saat dibaca.

## Tahap 1 — Menyiapkan library pada kernel notebook

Sel ini memakai Python dari **kernel yang dipilih**, memasang paket yang belum
sesuai, dan tidak membuat file program `.py`. Pasangan versi TorchVision/PyTorch
dipatok agar kode head dan checkpoint konsisten. Instalasi awal GPU besar dan
membutuhkan internet; paket yang sudah cocok tidak diunduh ulang.

Untuk **Windows NVIDIA RTX 2050**, biarkan `BUILD_PYTORCH = "cu126"`.
Pilih `"cu128"` untuk perangkat yang memerlukannya, atau `"cpu"` untuk menjalankan
tanpa GPU. Bila PyTorch sudah terpasang dengan build berbeda, pilih kernel/build
yang benar; penggantian PyTorch harus diaktifkan secara eksplisit.

In [ ]:
import sys
import os
import subprocess
import importlib
import importlib.metadata as metadata
from pathlib import Path

BUILD_PYTORCH = "cu126"  # "cu126", "cu128", atau "cpu"
PASANG_PAKET_JIKA_PERLU = True
IZINKAN_GANTI_PYTORCH = False  # True hanya jika sengaja mengganti build/versi kernel ini

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Pilih kernel Python 3.12 agar sesuai lingkungan yang diuji.")
if BUILD_PYTORCH not in {"cu126", "cu128", "cpu"}:
    raise ValueError("BUILD_PYTORCH harus cu126, cu128, atau cpu.")

def versi_paket(nama):
    try:
        return metadata.version(nama)
    except metadata.PackageNotFoundError:
        return None

print("Python kernel:", sys.executable)
pasangan = {"torch": "2.9.1", "torchvision": "0.24.1"}
torch_cocok = all(
    versi_paket(nama) is not None
    and versi_paket(nama).split("+")[0] == versi
    and ("+" + BUILD_PYTORCH) in versi_paket(nama)
    for nama, versi in pasangan.items()
)
if not torch_cocok:
    sudah_ada = any(versi_paket(nama) is not None for nama in pasangan)
    if sudah_ada and not IZINKAN_GANTI_PYTORCH:
        raise RuntimeError(
            f"Build kernel saat ini: torch={versi_paket('torch')}, "
            f"torchvision={versi_paket('torchvision')}. "
            "Pilih kernel/build yang sesuai, atau aktifkan IZINKAN_GANTI_PYTORCH "
            "untuk menggantinya, lalu Restart Kernel setelah pemasangan."
        )
    if not PASANG_PAKET_JIKA_PERLU:
        raise RuntimeError("PyTorch belum sesuai. Aktifkan PASANG_PAKET_JIKA_PERLU.")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "torch==2.9.1", "torchvision==0.24.1",
        "--index-url", f"https://download.pytorch.org/whl/{BUILD_PYTORCH}",
    ], check=True)
    importlib.invalidate_caches()
    raise RuntimeError("PyTorch selesai dipasang. Klik Restart Kernel, lalu jalankan sel dari awal.")

paket_penelitian = {
    "numpy": "2.2.6", "Pillow": "12.3.0", "matplotlib": "3.10.8",
    "torchmetrics": "1.9.0", "pycocotools": "2.0.11", "ipywidgets": "8.1.8",
}
perlu = [f"{nama}=={versi}" for nama, versi in paket_penelitian.items()
         if versi_paket(nama) != versi]
if perlu:
    if not PASANG_PAKET_JIKA_PERLU:
        raise RuntimeError(f"Paket belum sesuai: {perlu}")
    sudah_diimpor = any(nama in sys.modules for nama in ["numpy", "PIL", "matplotlib", "torchmetrics"])
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *perlu], check=True)
    importlib.invalidate_caches()
    if sudah_diimpor:
        raise RuntimeError("Library diperbarui. Klik Restart Kernel lalu jalankan dari awal.")
print("Library siap. Lanjut ke sel import.")


In [ ]:
import csv
import gc
import hashlib
import io
import json
import math
import random
import uuid
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime
from functools import partial

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from torchvision.transforms import functional as F
from torchvision.ops import box_iou, nms
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights,
)
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("PyTorch:", torch.__version__)
import torchvision
print("TorchVision:", torchvision.__version__)
# Memeriksa operator native sebelum training panjang.
nms(torch.tensor([[0., 0., 10., 10.]]), torch.tensor([0.9]), 0.5)
print("Operator deteksi siap.")


## Tahap 2 — Konfigurasi penelitian dalam satu sel

Ubah `DATASET_DIR` bila lokasi ekspor Roboflow berbeda. Untuk path Windows gunakan
bentuk `Path(r"D:\KAMPUS\dataset-kakao")`. Path tersebut harus langsung berisi
`train`, `valid`/`val`, dan `test`.

SGD `lr=0.001`, momentum `0.9`, weight decay `0.0005`, dan batch `4` merupakan
baseline konservatif untuk fine-tuning, bukan hasil optimasi. Batch `2` dapat
dipilih bila VRAM terbatas. **100 epoch adalah anggaran awal**, bukan jaminan semua
biji terdeteksi; model terbaik dipilih berdasarkan mAP validation, bukan epoch terakhir.
Tidak ada scheduler atau augmentasi warna ekstrem pada baseline ini.

Confidence `0.50` dipakai untuk PR/F1 awal. Nanti threshold operasional dipilih
dari validation. mAP selalu memakai kandidat skor rendah `0.001`, sehingga kurva
precision–recall tidak terpotong oleh confidence operasional.

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_ROOT = PROJECT_DIR / "hasil_penelitian"
os.environ["TORCH_HOME"] = str(PROJECT_DIR / ".cache" / "torch")

CLASS_NAMES = {0: "background", 1: "Tidak Fermentasi",
               2: "Setengah Fermentasi", 3: "Full Fermentasi"}
NUM_CLASSES = 4  # tiga kelas objek + background
# Tambah hanya jika nama ekspor berbeda DAN arti kelasnya memang sesuai.
CATEGORY_ALIASES = {}  # contoh: {"unfermented": 1}
MODEL_NAME = "ssdlite320_mobilenet_v3_large"
USE_PRETRAINED = True
NUM_EPOCHS = 100
BATCH_SIZE = 4
LR = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
RANDOM_SEED = 42
FLIP_PROB = 0.5
CONFIDENCE_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50
MAP_SCORE_THRESHOLD = 0.001
NMS_IOU_THRESHOLD = 0.55
MAX_DETECTIONS_PER_IMAGE = 300
COCO_MAX_DETECTIONS = (1, 10, 100)  # batas standar COCO mAP
# None = eksperimen baru. Isi hanya untuk melanjutkan last_model.pth run notebook ini.
RESUME_CHECKPOINT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if BUILD_PYTORCH != "cpu" and DEVICE.type != "cuda":
    raise RuntimeError("Build GPU dipilih tetapi CUDA tidak siap. Periksa kernel/driver, "
                       "atau pilih BUILD_PYTORCH='cpu' bila sengaja memakai CPU.")
torch.set_num_threads(min(4, os.cpu_count() or 1))
print("Folder kerja:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training memakai CPU; umumnya lebih lambat daripada GPU.")


## Tahap 3 — Membaca format COCO dan memeriksa dataset

**Input:** gambar serta `_annotations.coco.json` pada setiap split.
**Output:** dataset yang valid dan laporan jumlah gambar/objek per kelas.

ID kategori COCO tidak langsung dijadikan label model. Nama kategori dipetakan
ke label `1/2/3`; kategori induk Roboflow tanpa anotasi hanya dilaporkan. Nama
beranotasi yang belum dikenal menghentikan proses agar kelas tidak salah diam-diam.
COCO `[x, y, width, height]` diubah menjadi PyTorch `[xmin, ymin, xmax, ymax]`.

Audit memeriksa referensi ID, file, ukuran gambar, box, dan gambar identik antar
split. Data asli tidak diubah. Audit metadata test memeriksa integritas saja;
test tidak ikut training, pemilihan epoch, maupun tuning threshold.

In [ ]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset belum ditemukan di {DATASET_DIR}. Ubah DATASET_DIR pada konfigurasi.")
annotation_preview = DATASET_DIR / "train" / "_annotations.coco.json"
with annotation_preview.open(encoding="utf-8-sig") as file:
    coco_preview = json.load(file)
print("Kategori asli pada ekspor Roboflow:")
display(coco_preview.get("categories", []))
print("Jumlah gambar train:", len(coco_preview.get("images", [])))
print("Jumlah anotasi train:", len(coco_preview.get("annotations", [])))
del coco_preview


In [ ]:
# Semua definisi dataset berada di notebook ini; tidak mengimpor src/*.py.
import math
import re


def normalize_category_name(name):
    """Nama kelas tetap sama meski spasi, huruf besar, '_' atau '-' berbeda."""
    if not isinstance(name, str) or not name.strip():
        raise ValueError("Nama kategori harus berupa teks yang tidak kosong.")
    return re.sub(r"[\s_-]+", " ", name.strip().casefold())


def _integer(value, description):
    if isinstance(value, bool) or not isinstance(value, int):
        raise ValueError(f"{description} harus bilangan bulat: {value!r}")
    return value


def _coco_records(records, description):
    if not isinstance(records, list):
        raise ValueError(f"Bagian COCO '{description}' harus berupa list.")
    result = {}
    for item in records:
        if not isinstance(item, dict):
            raise ValueError(f"Setiap entri {description} harus object JSON.")
        item_id = _integer(item.get("id"), f"ID {description}")
        if item_id in result:
            raise ValueError(f"ID duplikat pada {description}: {item_id}")
        result[item_id] = item
    return result


def _class_aliases():
    # Label 0 khusus background dan tidak boleh menjadi anotasi objek.
    aliases = {normalize_category_name(name): label
               for label, name in CLASS_NAMES.items() if label > 0}
    for name, label in CATEGORY_ALIASES.items():
        label = _integer(label, f"Label alias {name!r}")
        if label not in CLASS_NAMES or label <= 0:
            raise ValueError(f"Alias {name!r} harus menunjuk kelas 1, 2, atau 3.")
        normalized = normalize_category_name(name)
        if normalized in aliases and aliases[normalized] != label:
            raise ValueError(f"Pemetaan alias bertentangan: {name!r}")
        aliases[normalized] = label
    return aliases


class CocoaDataset(Dataset):
    """Ekspor COCO dibaca langsung. File gambar/anotasi asli tidak diubah.

    Resize dan normalisasi dikerjakan transform internal SSDLite ketika forward.
    Dataset hanya mengubah PIL RGB menjadi tensor 0..1 dan flip train secara acak.
    """
    def __init__(self, images_dir, annotation_file, augment=False):
        self.images_dir = Path(images_dir).expanduser().resolve()
        self.annotation_file = Path(annotation_file).expanduser().resolve()
        self.augment = bool(augment)
        if not 0 <= FLIP_PROB <= 1:
            raise ValueError("FLIP_PROB harus berada antara 0 dan 1.")
        if not self.annotation_file.is_file():
            raise FileNotFoundError(f"Anotasi COCO tidak ditemukan: {self.annotation_file}")
        with self.annotation_file.open(encoding="utf-8-sig") as handle:
            coco = json.load(handle)
        if not isinstance(coco, dict):
            raise ValueError("Anotasi COCO harus object JSON.")
        self.images = _coco_records(coco.get("images"), "images")
        categories = _coco_records(coco.get("categories"), "categories")
        annotations = _coco_records(coco.get("annotations"), "annotations")
        self.ids = list(self.images)
        self.category_names = {cid: item.get("name") for cid, item in categories.items()}
        normalized = {cid: normalize_category_name(name)
                      for cid, name in self.category_names.items()}
        aliases = _class_aliases()
        self.category_mapping = {cid: aliases[name] for cid, name in normalized.items()
                                 if name in aliases}
        self.image_paths = []
        self.annotations_by_image = {image_id: [] for image_id in self.ids}
        self.class_counts = Counter({label: 0 for label in CLASS_NAMES if label > 0})
        seen_paths = set()
        for image_id in self.ids:
            record = self.images[image_id]
            width = _integer(record.get("width"), f"width gambar {image_id}")
            height = _integer(record.get("height"), f"height gambar {image_id}")
            if width <= 0 or height <= 0:
                raise ValueError(f"Ukuran gambar {image_id} harus positif.")
            filename = record.get("file_name")
            if not isinstance(filename, str) or not filename.strip():
                raise ValueError(f"Nama file gambar {image_id} tidak valid.")
            # Backslash Windows diubah menjadi separator portabel. Path keluar ditolak.
            path = (self.images_dir / filename.replace("\\", "/")).resolve()
            if not path.is_relative_to(self.images_dir):
                raise ValueError(f"Path gambar keluar folder split: {filename!r}")
            if path in seen_paths:
                raise ValueError(f"File yang sama dirujuk dua image_id: {filename!r}")
            if not path.is_file():
                raise FileNotFoundError(f"Gambar tidak ditemukan: {path}")
            try:
                with Image.open(path) as image:
                    actual_size = image.size
                    image.verify()
            except (OSError, ValueError) as error:
                raise ValueError(f"Gambar rusak/tidak bisa dibaca: {path}") from error
            if actual_size != (width, height):
                raise ValueError(f"Dimensi {filename!r} berbeda: JSON={(width, height)}, "
                                 f"file={actual_size}. Perbaiki ekspor terlebih dahulu.")
            self.image_paths.append(path)
            seen_paths.add(path)

        used_categories = set()
        for annotation_id, record in annotations.items():
            image_id = _integer(record.get("image_id"), f"image_id anotasi {annotation_id}")
            category_id = _integer(record.get("category_id"), f"category_id anotasi {annotation_id}")
            if image_id not in self.images:
                raise ValueError(f"Anotasi {annotation_id} merujuk image_id yang tidak ada.")
            if category_id not in categories:
                raise ValueError(f"Anotasi {annotation_id} merujuk kategori yang tidak ada.")
            if category_id not in self.category_mapping:
                raise ValueError(f"Kategori beranotasi belum dipetakan: "
                                 f"{self.category_names[category_id]!r} (ID {category_id}). "
                                 "Periksa CLASS_NAMES/CATEGORY_ALIASES, jangan abaikan objek.")
            bbox = record.get("bbox")
            if not isinstance(bbox, list) or len(bbox) != 4:
                raise ValueError(f"bbox anotasi {annotation_id} harus [x, y, lebar, tinggi].")
            if any(isinstance(v, bool) or not isinstance(v, (int, float)) for v in bbox):
                raise ValueError(f"bbox anotasi {annotation_id} harus berisi empat angka.")
            x, y, width, height = map(float, bbox)
            image_info = self.images[image_id]
            if (not all(math.isfinite(v) for v in (x, y, width, height))
                    or x < 0 or y < 0 or width <= 0 or height <= 0
                    or x + width > image_info["width"]
                    or y + height > image_info["height"]):
                raise ValueError(f"bbox anotasi {annotation_id} tidak valid/keluar gambar: {bbox}")
            crowd = _integer(record.get("iscrowd", 0), f"iscrowd anotasi {annotation_id}")
            if crowd not in (0, 1):
                raise ValueError(f"iscrowd anotasi {annotation_id} harus 0 atau 1.")
            # SSDLite tidak memperlakukan crowd khusus saat menghitung training loss.
            if self.augment and crowd:
                raise ValueError("Anotasi crowd ditemukan pada train. Untuk menghitung biji "
                                 "satu per satu, beri bounding box setiap biji (iscrowd=0).")
            label = self.category_mapping[category_id]
            self.annotations_by_image[image_id].append({
                "boxes": [x, y, x + width, y + height], "labels": label,
                "area": width * height, "iscrowd": crowd,
            })
            self.class_counts[label] += 1
            used_categories.add(category_id)
        # Roboflow kadang menambahkan kategori induk ID 0 tanpa anotasi; aman dilaporkan.
        self.unused_categories = [{"id": cid, "name": self.category_names[cid]}
                                  for cid in categories if cid not in used_categories]

    def __len__(self):
        return len(self.ids)

    def _load_sample(self, index):
        image_id = self.ids[index]
        with Image.open(self.image_paths[index]) as image:
            image = image.convert("RGB")
        objects = self.annotations_by_image[image_id]
        target = {
            "boxes": torch.tensor([obj["boxes"] for obj in objects],
                                  dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor([obj["labels"] for obj in objects], dtype=torch.int64),
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": torch.tensor([obj["area"] for obj in objects], dtype=torch.float32),
            "iscrowd": torch.tensor([obj["iscrowd"] for obj in objects], dtype=torch.int64),
        }
        return F.to_tensor(image), target

    def __getitem__(self, index):
        image, target = self._load_sample(index)
        if self.augment and torch.rand(()).item() < FLIP_PROB:
            image, target = horizontal_flip(image, target)
        return image, target


def horizontal_flip(image, target):
    """Bounding box bergerak bersama gambar; label dan area tetap sama."""
    width = image.shape[-1]
    flipped_target = {key: value.clone() for key, value in target.items()}
    boxes = flipped_target["boxes"]
    boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
    return F.hflip(image), flipped_target


def detection_collate(batch):
    # Jumlah objek tiap gambar berbeda, sehingga target tidak bisa ditumpuk biasa.
    images, targets = zip(*batch)
    return list(images), list(targets)


In [ ]:
def resolve_split(root, split):
    root = Path(root).expanduser().resolve()
    split = "val" if split in ("valid", "validation") else split
    if split not in ("train", "val", "test"):
        raise ValueError(f"Split tidak dikenal: {split}")
    names = ("val", "valid") if split == "val" else (split,)
    present = [root / name for name in names if (root / name).is_dir()]
    if len(present) > 1:
        raise ValueError("Ada folder val dan valid. Sisakan satu split validation agar tidak ambigu.")
    if not present:
        raise FileNotFoundError(f"Folder split {split} tidak ditemukan dalam {root}")
    return present[0]


def audit_dataset(root):
    """Audit metadata test hanya untuk integritas; test tidak dipakai memilih model."""
    datasets, report = {}, {"status": "passed", "splits": {}, "warnings": []}
    seen_hashes, seen_annotation_paths = {}, set()
    for split in ("train", "val", "test"):
        split_dir = resolve_split(root, split)
        dataset = CocoaDataset(split_dir, split_dir / "_annotations.coco.json",
                               augment=(split == "train"))
        if len(dataset) < (2 if split == "train" else 1):
            raise ValueError(f"Split {split}: minimal 2 gambar train dan 1 gambar val/test.")
        if dataset.annotation_file in seen_annotation_paths:
            raise ValueError("File anotasi dipakai ulang pada split berbeda.")
        seen_annotation_paths.add(dataset.annotation_file)
        if split in ("train", "val"):
            noncrowd = sum(obj["iscrowd"] == 0
                           for objects in dataset.annotations_by_image.values() for obj in objects)
            if not noncrowd:
                raise ValueError(f"Split {split} tidak memiliki objek non-crowd untuk training/validasi.")
        # ID COCO boleh berbeda antar split: nama kelas selalu dipetakan ke label model yang sama.
        local_hashes = defaultdict(list)
        for path in dataset.image_paths:
            digest = hashlib.sha256()
            with path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(chunk)
            digest = digest.hexdigest()
            if digest in seen_hashes and seen_hashes[digest][0] != split:
                other_split, other_name = seen_hashes[digest]
                raise ValueError(f"Data leakage: isi gambar identik pada {other_split}/{other_name} "
                                 f"dan {split}/{path.name}. Pisahkan ulang sumber data.")
            seen_hashes[digest] = (split, path.name)
            local_hashes[digest].append(path.name)
        duplicate_groups = [names for names in local_hashes.values() if len(names) > 1]
        missing_classes = [CLASS_NAMES[label] for label, count in dataset.class_counts.items() if count == 0]
        if duplicate_groups:
            report["warnings"].append(f"{split}: ada gambar duplikat dalam split; periksa laporan.")
        if missing_classes:
            report["warnings"].append(f"{split}: kelas tanpa objek: {', '.join(missing_classes)}.")
        report["splits"][split] = {
            "images": len(dataset), "objects": sum(dataset.class_counts.values()),
            "images_without_objects": sum(not rows for rows in dataset.annotations_by_image.values()),
            "class_counts": {CLASS_NAMES[label]: count for label, count in dataset.class_counts.items()},
            "category_mapping": {str(cid): {"name": dataset.category_names[cid], "label": label}
                                 for cid, label in dataset.category_mapping.items()},
            "unused_categories": dataset.unused_categories,
            "duplicate_images_within_split": duplicate_groups,
            "annotation_file": str(dataset.annotation_file),
        }
        datasets[split] = dataset
    report["limitations"] = ("SHA-256 mendeteksi file identik, bukan foto mirip/crop/encoding berbeda. "
                             "Gambar dari pengambilan yang sama harus berada di split yang sama. "
                             "Augmentasi hanya train; warna tidak diubah karena berkaitan dengan fermentasi.")
    return datasets, report


def show_dataset_examples(dataset, n=3, force_flip=False):
    """Tampilkan gambar asli atau flip pasti untuk menjelaskan pergeseran bbox."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    n = min(int(n), len(dataset))
    if n < 1:
        raise ValueError("Dataset kosong; tidak ada contoh untuk ditampilkan.")
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
    colors = {1: "#d62728", 2: "#e3a008", 3: "#238636"}
    for index, axis in enumerate(axes.ravel()):
        image, target = dataset._load_sample(index)  # tidak memakai flip acak saat membandingkan
        if force_flip:
            image, target = horizontal_flip(image, target)
        axis.imshow(image.permute(1, 2, 0).numpy())
        for box, label in zip(target["boxes"].tolist(), target["labels"].tolist()):
            x1, y1, x2, y2 = box
            color = colors[label]
            axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     fill=False, edgecolor=color, linewidth=1.5))
            axis.text(x1, max(0, y1 - 2), CLASS_NAMES[label], fontsize=7, color="white",
                      bbox={"facecolor": color, "alpha": 0.8, "pad": 1})
        axis.set_title(f"{'Flip horizontal' if force_flip else 'Asli'} — {len(target['labels'])} objek")
        axis.axis("off")
    fig.tight_layout()
    plt.show()
    return fig


In [ ]:
datasets, dataset_report = audit_dataset(DATASET_DIR)
for split, values in dataset_report["splits"].items():
    print(f"{split}: {values['images']} gambar, {values['objects']} biji | {values['class_counts']}")
for message in dataset_report["warnings"]:
    print("Periksa:", message)
print("Audit selesai. Laporan lengkap akan disimpan bersama hasil eksperimen.")


## Tahap 4 — Preprocessing, augmentasi, dan DataLoader

Gambar dibaca sebagai RGB lalu menjadi tensor float32 `[C,H,W]` dalam `[0,1]`.
**Resize 320×320 dan normalisasi dilakukan di dalam model SSDLite** bersama
penyesuaian box. Jangan menambah normalisasi atau resize kedua kali di loader.

Training memakai horizontal flip dengan peluang 50% setiap gambar dibaca.
Box ikut berubah: `xmin_baru = W - xmax_lama`, `xmax_baru = W - xmin_lama`.
Ini augmentasi **online**: jumlah file tetap; variasi muncul saat epoch berjalan.
Validation dan test tidak di-flip. Dua tampilan berikut memakai gambar yang sama;
contoh kedua sengaja flip 100% untuk menjelaskan perubahan box, bukan mengubah
probabilitas training.

In [ ]:
original_figure = show_dataset_examples(datasets["train"], n=3, force_flip=False)
flipped_figure = show_dataset_examples(datasets["train"], n=3, force_flip=True)
plt.close(original_figure)
plt.close(flipped_figure)


In [ ]:
# num_workers=0 aman untuk Windows/Jupyter; pin_memory=False menghindari
# masalah CUDA resource already mapped yang pernah muncul pada Windows.
train_preview_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=0, pin_memory=False,
                                 collate_fn=detection_collate)
sample_images, sample_targets = next(iter(train_preview_loader))
print("Gambar batch:", len(sample_images))
print("Tensor gambar:", sample_images[0].shape, sample_images[0].dtype)
print("boxes:", sample_targets[0]["boxes"].shape, sample_targets[0]["boxes"].dtype)
print("labels:", sample_targets[0]["labels"].dtype)
print("Target:", list(sample_targets[0]))
del sample_images, sample_targets, train_preview_loader


## Tahap 5 — SSDLite-MobileNetV3-Large dan transfer learning

Model resmi: `torchvision.models.detection.ssdlite320_mobilenet_v3_large`.
Fungsi `build_model(pretrained=True)` memuat detection weights COCO, kemudian
mengganti **classification head** menjadi empat kelas. Backbone MobileNetV3,
feature blocks, dan bbox regression head mempertahankan bobot pretrained.

Tidak memasukkan `num_classes=4` saat memuat bobot COCO karena bobot tersebut
memiliki jumlah kelas asalnya sendiri. Head kelas diganti **setelah** pemuatan.
`pretrained=False` disediakan untuk memuat checkpoint lengkap tanpa mengunduh
bobot lagi; training eksperimen baru tetap memakai `USE_PRETRAINED=True`.

In [ ]:
def build_model(pretrained=True):
    """SSDLite320 + MobileNetV3-Large; tiga kelas biji dan background.

    Eksperimen baru menggunakan transfer learning COCO. Ini bukan melanjutkan
    checkpoint skripsi lama. pretrained=False dipakai saat memuat checkpoint
    sendiri agar tidak mengunduh bobot lain.
    """
    if NUM_CLASSES != 4 or set(CLASS_NAMES) != {0, 1, 2, 3}:
        raise ValueError("Diperlukan 3 kelas fermentasi + background (4 kelas).")
    if MODEL_NAME != "ssdlite320_mobilenet_v3_large":
        raise ValueError("Arsitektur penelitian harus ssdlite320_mobilenet_v3_large.")
    options = dict(
        weights_backbone=None,
        score_thresh=MAP_SCORE_THRESHOLD,
        nms_thresh=NMS_IOU_THRESHOLD,
        detections_per_img=MAX_DETECTIONS_PER_IMAGE,
        progress=True,
    )
    if not pretrained:
        return ssdlite320_mobilenet_v3_large(weights=None, num_classes=NUM_CLASSES, **options)
    model = ssdlite320_mobilenet_v3_large(
        weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT, **options,
    )
    old_head = model.head.classification_head
    in_channels = [block[0][0].in_channels for block in old_head.module_list]
    anchors = model.anchor_generator.num_anchors_per_location()
    # Ganti head kelas saja; backbone dan regresi box tetap pretrained COCO.
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels, anchors, NUM_CLASSES,
        partial(nn.BatchNorm2d, eps=0.001, momentum=0.03),
    )
    return model


def collate_fn(batch):
    # Ukuran gambar dan jumlah bounding box tiap gambar boleh berbeda.
    return tuple(zip(*batch))


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def capture_rng_state():
    state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": (state[0], state[1].tolist(), *state[2:]),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    value = state["numpy"]
    np.random.set_state((value[0], np.array(value[1], dtype=np.uint32), *value[2:]))
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda"):
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])


def save_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, allow_nan=False)
    temporary.replace(path)


def save_checkpoint_atomic(path, checkpoint):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def load_trusted_checkpoint(path, device="cpu"):
    """Muat hanya checkpoint buatan sendiri/sumber yang Anda percaya.

    File .pth menggunakan serialisasi Python. Jangan memuat file .pth acak.
    Pemeriksaan metadata mencegah pertukaran arsitektur/urutan kelas.
    """
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint belum ada: {path}")
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
        raise ValueError("Checkpoint harus memiliki model_state_dict.")
    if checkpoint.get("model_name") != MODEL_NAME or checkpoint.get("num_classes") != NUM_CLASSES:
        raise ValueError("Arsitektur atau jumlah kelas checkpoint tidak cocok.")
    saved_classes = {int(key): value for key, value in checkpoint.get("class_names", {}).items()}
    if saved_classes != CLASS_NAMES:
        raise ValueError("Nama/urutan kelas checkpoint tidak cocok dengan notebook.")
    return checkpoint


def dataset_signature(datasets):
    """Sidik data: anotasi, pemetaan label, nama dan isi berkas gambar."""
    digest = hashlib.sha256()
    for split, dataset in sorted(datasets.items()):
        annotation_file = Path(dataset.annotation_file)
        images_dir = Path(dataset.images_dir).resolve()
        digest.update(split.encode("utf-8"))
        digest.update(annotation_file.read_bytes())
        digest.update(json.dumps(dataset.category_mapping, sort_keys=True).encode("utf-8"))
        coco = json.loads(annotation_file.read_text(encoding="utf-8-sig"))
        for record in sorted(coco["images"], key=lambda item: str(item["file_name"])):
            relative_name = str(record["file_name"]).replace("\\", "/")
            image_path = (images_dir / relative_name).resolve()
            if not image_path.is_relative_to(images_dir):
                raise ValueError(f"Nama gambar berada di luar folder dataset: {relative_name}")
            digest.update(relative_name.encode("utf-8"))
            with image_path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(chunk)
    return digest.hexdigest()


@contextmanager
def singleton_batch_norm(model, batch_size):
    # Batch terakhir berisi satu gambar tetap dilatih. BN menggunakan running
    # statistics sementara karena feature map 1x1 tidak cukup untuk variansi.
    changed = []
    if batch_size == 1:
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm) and module.training:
                module.eval()
                changed.append(module)
    try:
        yield
    finally:
        for module in changed:
            module.train()


def save_training_history(metrics_dir, history):
    metrics_dir = Path(metrics_dir)
    save_json_atomic(metrics_dir / "training_history.json", history)
    if history:
        target = metrics_dir / "training_history.csv"
        temporary = target.with_name(target.name + ".tmp")
        with temporary.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(history[0]))
            writer.writeheader()
            writer.writerows(history)
        temporary.replace(target)


## Tahap 6 — Evaluasi object detection

**IoU** mengukur luas irisan box dibagi luas gabungannya. Pada confidence
operasional, prediksi diurutkan berdasarkan skor. Prediksi menjadi **TP** bila
kelasnya sama dan IoU ≥ 0.50 dengan satu anotasi yang belum dipasangkan.
Prediksi yang tidak cocok menjadi **FP**; anotasi yang belum terdeteksi menjadi
**FN**. Duplikat tidak boleh memakai anotasi yang sama dua kali.

Precision = TP/(TP+FP), Recall = TP/(TP+FN), F1 = 2PR/(P+R).
AP/mAP dihitung oleh TorchMetrics/pycocotools dari rentang confidence, dengan
mAP@0.5 serta mAP@0.5:0.95. Batas pelaporan COCO tetap 100 deteksi per gambar;
PR/F1 operasional memakai semua kandidat model yang lolos confidence. Nilai AP
kelas tanpa anotasi adalah `None`, bukan performa nol yang seolah terukur.
Rata-rata IoU hanya mencakup TP, sehingga harus dibaca bersama recall.

In [ ]:
def _number_or_none(value):
    """COCO memakai -1 untuk AP yang tidak terdefinisi, bukan AP nol."""
    number = float(value)
    return number if math.isfinite(number) and number >= 0 else None


def _rates(tp: int, fp: int, fn: int) -> dict:
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "support": tp + fn,
        "predicted_count": tp + fp,
    }


def _intersection_over_prediction(prediction_box, crowd_boxes):
    """COCO memperlakukan satu box crowd sebagai area yang boleh diabaikan."""
    top_left = torch.maximum(prediction_box[:2], crowd_boxes[:, :2])
    bottom_right = torch.minimum(prediction_box[2:], crowd_boxes[:, 2:])
    intersection = (bottom_right - top_left).clamp(min=0).prod(dim=1)
    prediction_area = (prediction_box[2:] - prediction_box[:2]).clamp(min=0).prod()
    return intersection / prediction_area.clamp(min=torch.finfo(torch.float32).eps)


def _match_image(prediction, target, confidence_threshold, iou_threshold, class_ids):
    """Hitung TP/FP/FN pada satu gambar; box salah kelas menjadi FP dan FN."""
    counts = {label: {"tp": 0, "fp": 0, "fn": 0, "ignored": 0} for label in class_ids}
    matched_ious = []
    crowds = target.get("iscrowd", torch.zeros_like(target["labels"])).bool()

    for label in class_ids:
        ordinary_boxes = target["boxes"][(target["labels"] == label) & ~crowds]
        crowd_boxes = target["boxes"][(target["labels"] == label) & crowds]
        keep = (prediction["labels"] == label) & (prediction["scores"] >= confidence_threshold)
        boxes = prediction["boxes"][keep]
        scores = prediction["scores"][keep]
        # Stable sorting makes tied confidences reproducible in input order.
        boxes = boxes[torch.argsort(scores, descending=True, stable=True)]
        used = torch.zeros(len(ordinary_boxes), dtype=torch.bool)
        overlaps = box_iou(boxes, ordinary_boxes)

        for prediction_index, prediction_box in enumerate(boxes):
            if len(ordinary_boxes):
                candidates = overlaps[prediction_index].clone()
                candidates[used] = -1
                best_iou, ground_truth_index = candidates.max(dim=0)
                if float(best_iou) >= iou_threshold:
                    used[ground_truth_index] = True
                    counts[label]["tp"] += 1
                    matched_ious.append(float(best_iou))
                    continue

            # Crowd is excluded from support/FN. An unmatched same-class
            # prediction inside crowd is ignored, not counted as a false positive.
            if len(crowd_boxes) and bool(
                (_intersection_over_prediction(prediction_box, crowd_boxes) >= iou_threshold).any()
            ):
                counts[label]["ignored"] += 1
            else:
                counts[label]["fp"] += 1
        counts[label]["fn"] = int((~used).sum())
    return counts, matched_ious


def _cpu_prediction(prediction, class_ids):
    result = {
        "boxes": prediction["boxes"].detach().to("cpu", dtype=torch.float32).reshape(-1, 4),
        "labels": prediction["labels"].detach().to("cpu", dtype=torch.int64).reshape(-1),
        "scores": prediction["scores"].detach().to("cpu", dtype=torch.float32).reshape(-1),
    }
    if len(result["boxes"]) != len(result["labels"]) or len(result["boxes"]) != len(result["scores"]):
        raise ValueError("Jumlah boxes, labels, dan scores prediksi harus sama.")
    unknown = set(result["labels"].tolist()) - set(class_ids)
    if unknown:
        raise ValueError(f"Label prediksi tidak dikenal: {sorted(unknown)}.")
    return result


def _cpu_target(target, class_ids):
    boxes = target["boxes"].detach().to("cpu", dtype=torch.float32).reshape(-1, 4)
    labels = target["labels"].detach().to("cpu", dtype=torch.int64).reshape(-1)
    result = {"boxes": boxes, "labels": labels}
    result["area"] = (
        target["area"].detach().to("cpu", dtype=torch.float32).reshape(-1)
        if "area" in target
        else (boxes[:, 2:] - boxes[:, :2]).clamp(min=0).prod(dim=1)
    )
    result["iscrowd"] = (
        target["iscrowd"].detach().to("cpu", dtype=torch.int64).reshape(-1)
        if "iscrowd" in target
        else torch.zeros_like(labels)
    )
    if any(len(result[key]) != len(boxes) for key in ("labels", "area", "iscrowd")):
        raise ValueError("Jumlah boxes, labels, area, dan iscrowd anotasi harus sama.")
    unknown = set(labels.tolist()) - set(class_ids)
    if unknown:
        raise ValueError(f"Label anotasi tidak dikenal: {sorted(unknown)}.")
    return result


In [ ]:
@torch.inference_mode()
def evaluate_model(model, data_loader, device, confidence_threshold=0.5, iou_threshold=0.5):
    """Kembalikan metrik Python biasa yang aman disimpan sebagai JSON.

    Tidak ada ground truth kelas tertentu: support=0, recall/F1=0 dan AP=None.
    AP=None juga berlaku untuk kelas yang hanya mempunyai anotasi crowd.
    mean_matched_iou hanya mencakup TP non-crowd pada threshold operasional.
    """
    confidence_threshold = float(confidence_threshold)
    iou_threshold = float(iou_threshold)
    if not math.isfinite(confidence_threshold) or not 0 <= confidence_threshold <= 1:
        raise ValueError("Confidence threshold harus berada pada rentang 0 sampai 1.")
    if not math.isfinite(iou_threshold) or not 0 < iou_threshold <= 1:
        raise ValueError("IoU threshold harus lebih dari 0 dan paling besar 1.")

    class_ids = sorted(label for label in CLASS_NAMES if label != 0)
    counts = {label: {"tp": 0, "fp": 0, "fn": 0, "ignored": 0} for label in class_ids}
    matched_ious = []
    evaluated_images = 0
    metric = MeanAveragePrecision(
        box_format="xyxy",
        class_metrics=True,
        extended_summary=True,
        max_detection_thresholds=list(COCO_MAX_DETECTIONS),
    )
    # Standar COCO tetap memakai maksimum 100 deteksi per gambar. Model dapat
    # menghasilkan lebih banyak kandidat; hanya peringatan berulang yang dimatikan,
    # bukan batas mAP. Precision/recall/F1 di bawah memakai semua kandidat lolos ambang.
    metric.warn_on_many_detections = False

    was_training = model.training
    previous_score_threshold = getattr(model, "score_thresh", None)
    # TorchVision SSD filters scores inside forward(). Keep low scores for AP;
    # the operational PR filter happens later and never truncates the AP input.
    if previous_score_threshold is not None:
        model.score_thresh = min(MAP_SCORE_THRESHOLD, confidence_threshold)
    model.eval()
    try:
        for images, targets in data_loader:
            predictions = model([image.to(device) for image in images])
            if len(predictions) != len(targets):
                raise ValueError("Model harus mengembalikan satu prediksi per gambar.")
            cpu_predictions = [_cpu_prediction(prediction, class_ids) for prediction in predictions]
            cpu_targets = [_cpu_target(target, class_ids) for target in targets]
            map_predictions = []
            for prediction, target in zip(cpu_predictions, cpu_targets):
                keep = prediction["scores"] >= MAP_SCORE_THRESHOLD
                map_predictions.append({key: value[keep] for key, value in prediction.items()})
                image_counts, image_ious = _match_image(
                    prediction, target, confidence_threshold, iou_threshold, class_ids
                )
                for label in class_ids:
                    for key in counts[label]:
                        counts[label][key] += image_counts[label][key]
                matched_ious.extend(image_ious)
            if cpu_targets:
                metric.update(map_predictions, cpu_targets)
                evaluated_images += len(cpu_targets)
        raw_map = metric.compute() if evaluated_images else None
    finally:
        if previous_score_threshold is not None:
            model.score_thresh = previous_score_threshold
        model.train(was_training)

    per_class_ap = {}
    if raw_map is not None:
        observed_classes = raw_map["classes"].reshape(-1).tolist()
        average_precisions = raw_map["map_per_class"].reshape(-1)
        iou_50_index = next(
            index for index, threshold in enumerate(metric.iou_thresholds) if abs(threshold - 0.5) < 1e-6
        )
        for class_index, label in enumerate(observed_classes):
            # Public extended summary: [IoU, recall, class, area, maxDet].
            # Area index 0 means all sizes; last maxDet is the COCO limit 100.
            precision_50 = raw_map["precision"][iou_50_index, :, class_index, 0, -1]
            valid_precision = precision_50[precision_50 >= 0]
            per_class_ap[int(label)] = {
                "ap": _number_or_none(average_precisions[class_index]),
                "ap_50": _number_or_none(valid_precision.mean()) if len(valid_precision) else None,
            }

    per_class = {}
    for label in class_ids:
        values = counts[label]
        per_class[str(label)] = {
            "name": CLASS_NAMES[label],
            **_rates(values["tp"], values["fp"], values["fn"]),
            "ignored": values["ignored"],
            **per_class_ap.get(label, {"ap": None, "ap_50": None}),
        }
    total = {key: sum(values[key] for values in counts.values()) for key in ("tp", "fp", "fn", "ignored")}
    return {
        "map": _number_or_none(raw_map["map"]) if raw_map is not None else None,
        "map_50": _number_or_none(raw_map["map_50"]) if raw_map is not None else None,
        "map_75": _number_or_none(raw_map["map_75"]) if raw_map is not None else None,
        **_rates(total["tp"], total["fp"], total["fn"]),
        "ignored": total["ignored"],
        "per_class": per_class,
        "mean_matched_iou": sum(matched_ious) / len(matched_ious) if matched_ious else None,
        "evaluated_images": evaluated_images,
        "confidence_threshold": confidence_threshold,
        "iou_threshold": iou_threshold,
        "map_score_threshold": MAP_SCORE_THRESHOLD,
        "coco_max_detections": list(COCO_MAX_DETECTIONS),
        "crowd_policy": "Exclude crowd from PR support; ignore unmatched same-class predictions with intersection/prediction-area >= IoU threshold. COCO AP receives original crowd annotations.",
        "zero_support_policy": "PR denominators of zero return 0; undefined COCO AP returns null.",
    }


def show_metrics(metrics):
    """Tampilkan ringkasan dan tabel per kelas, tanpa memerlukan pandas."""
    from html import escape
    from IPython.display import HTML, display

    def formatted(value):
        return "Tidak terdefinisi" if value is None else f"{value:.4f}"

    print(f"Jumlah gambar: {metrics['evaluated_images']}")
    print(f"mAP@0.5:0.95: {formatted(metrics['map'])} | "
          f"mAP@0.5: {formatted(metrics['map_50'])}")
    print(f"Precision: {formatted(metrics['precision'])} | "
          f"Recall: {formatted(metrics['recall'])} | F1: {formatted(metrics['f1'])}")
    print(f"Rata-rata IoU pada true positive: {formatted(metrics['mean_matched_iou'])}")
    print(f"Confidence PR/F1: {metrics['confidence_threshold']:.2f}; "
          f"IoU pencocokan: {metrics['iou_threshold']:.2f}. "
          f"mAP memakai skor minimum {metrics['map_score_threshold']} "
          f"dan batas standar COCO {metrics['coco_max_detections'][-1]}.")

    headings = ["Kelas", "AP@0.5:0.95", "AP@0.5", "Precision", "Recall", "F1", "TP", "FP", "FN", "Jumlah anotasi"]
    header = "".join(f"<th>{escape(name)}</th>" for name in headings)
    rows = []
    for values in metrics["per_class"].values():
        columns = [values["name"], formatted(values["ap"]), formatted(values["ap_50"]),
                   formatted(values["precision"]), formatted(values["recall"]), formatted(values["f1"]),
                   values["tp"], values["fp"], values["fn"], values["support"]]
        rows.append("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in columns) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))


## Tahap 7 — Training eksperimen baru

Loss bawaan SSDLite mengembalikan `classification` dan `bbox_regression`.
Yang pertama melatih kelas/background; yang kedua melatih koordinat box.
Keduanya dijumlahkan untuk backpropagation. Tidak memakai loss klasifikasi
gambar biasa. Batch terakhir yang hanya satu gambar memakai running statistics
BatchNorm agar tetap dapat diproses tanpa membuang gambar tersebut.

**Menjalankan sel training di bawah memulai proses training** sesuai konfigurasi.
Unduhan detection weights COCO hanya diperlukan pertama kali. Setiap eksperimen
baru memperoleh folder waktu+ID sendiri; checkpoint lama tidak ditimpa.
`best_model.pth` dipilih berdasarkan **mAP@0.5:0.95 validation tertinggi**,
sedangkan `last_model.pth` menyimpan epoch terakhir yang selesai.

Checkpoint memuat bobot, optimizer, epoch, loss, history, konfigurasi, pemetaan
kelas, fingerprint train/val, serta keadaan random. CSV/JSON diperbarui setiap
epoch. Semua gambar train dipakai; test tidak dimasukkan ke loop training.

In [ ]:
def train_experiment(train_dataset, val_dataset, num_epochs=NUM_EPOCHS,
                     batch_size=BATCH_SIZE, output_root=OUTPUT_ROOT,
                     pretrained=True, resume=None):
    """Latih train, pilih best berdasarkan mAP validation; test tidak disentuh.

    num_epochs adalah TOTAL epoch, termasuk epoch yang sudah selesai saat
    resume. Eksperimen baru selalu membuat folder baru; resume hanya menerima
    last_model.pth dari eksperimen notebook ini.
    """
    if not isinstance(num_epochs, int) or num_epochs < 1:
        raise ValueError("Jumlah epoch harus bilangan bulat positif.")
    if not isinstance(batch_size, int) or batch_size < 2:
        raise ValueError("Gunakan batch size minimal 2. Untuk RTX 2050 mulai dari 4 atau 2.")
    if len(train_dataset) < 2 or len(val_dataset) < 1:
        raise ValueError("Training perlu minimal 2 gambar train dan 1 gambar validation.")
    if not (math.isfinite(LR) and LR > 0 and 0 <= MOMENTUM < 1 and WEIGHT_DECAY >= 0):
        raise ValueError("Konfigurasi learning rate/momentum/weight decay tidak valid.")
    device = torch.device(DEVICE)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA diminta tetapi belum tersedia. Periksa kernel Python dan build PyTorch.")
    # Kesalahan CUDA akan menghentikan proses, tanpa otomatis beralih ke CPU.
    signature = dataset_signature({"train": train_dataset, "val": val_dataset})
    hyperparameters = {
        "batch_size": batch_size, "learning_rate": LR, "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY, "optimizer": "SGD", "seed": RANDOM_SEED,
        "horizontal_flip_probability": FLIP_PROB,
        "map_score_threshold": MAP_SCORE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "max_detections_per_image": MAX_DETECTIONS_PER_IMAGE,
        "coco_max_detections": list(globals().get("COCO_MAX_DETECTIONS", (1, 10, 100))),
        "confidence_threshold": CONFIDENCE_THRESHOLD, "iou_threshold": IOU_THRESHOLD,
        "num_workers": 0, "pin_memory": False, "deterministic": True,
    }
    metadata = {
        "MODEL_NAME": MODEL_NAME, "NUM_CLASSES": NUM_CLASSES,
        "CLASS_NAMES": CLASS_NAMES, "LEARNING_RATE": LR, "MOMENTUM": MOMENTUM,
        "WEIGHT_DECAY": WEIGHT_DECAY, "BATCH_SIZE": batch_size, "NUM_EPOCHS": num_epochs,
        "RANDOM_SEED": RANDOM_SEED, "HORIZONTAL_FLIP_PROBABILITY": FLIP_PROB,
        "MAP_SCORE_THRESHOLD": MAP_SCORE_THRESHOLD, "NMS_IOU_THRESHOLD": NMS_IOU_THRESHOLD,
        "MAX_DETECTIONS_PER_IMAGE": MAX_DETECTIONS_PER_IMAGE,
        "CONFIDENCE_THRESHOLD": CONFIDENCE_THRESHOLD, "IOU_THRESHOLD": IOU_THRESHOLD,
        "NOTEBOOK": "KAKAO_TRAINING_DARI_NOL.ipynb",
    }
    start_epoch, best_map, best_epoch, history = 0, -1.0, 0, []
    resumed_checkpoint = None
    if resume is None:
        run_name = "eksperimen_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
        run_dir = Path(output_root).expanduser().resolve() / run_name
        run_dir.mkdir(parents=True, exist_ok=False)
    else:
        resume_path = Path(resume).expanduser().resolve()
        if resume_path.name != "last_model.pth" or resume_path.parent.name != "checkpoints":
            raise ValueError("Pilih last_model.pth di folder checkpoints eksperimen untuk resume.")
        resumed_checkpoint = load_trusted_checkpoint(resume_path, "cpu")
        required = {"optimizer_state_dict", "epoch", "history", "rng_state", "loader_rng_state",
                    "best_epoch", "best_map", "hyperparameters", "dataset_signature"}
        if not required <= resumed_checkpoint.keys():
            raise ValueError("Checkpoint tidak lengkap untuk resume; gunakan last_model.pth notebook ini.")
        if resumed_checkpoint["dataset_signature"] != signature:
            raise ValueError("Data/anotasi train atau validation berubah. Mulai eksperimen baru.")
        if resumed_checkpoint["hyperparameters"] != hyperparameters:
            raise ValueError("Hyperparameter resume berubah. Samakan konfigurasi dan batch size run awal.")
        start_epoch = int(resumed_checkpoint["epoch"])
        if num_epochs <= start_epoch:
            raise ValueError(f"Total epoch harus melebihi epoch yang sudah selesai ({start_epoch}).")
        history = list(resumed_checkpoint["history"])
        if len(history) != start_epoch or not history or int(history[-1]["epoch"]) != start_epoch:
            raise ValueError("Riwayat training tidak cocok dengan epoch checkpoint.")
        best_map = float(resumed_checkpoint["best_map"])
        best_epoch = int(resumed_checkpoint["best_epoch"])
        run_dir = resume_path.parent.parent
        best_saved = load_trusted_checkpoint(resume_path.parent / "best_model.pth", "cpu")
        if (best_saved.get("dataset_signature") != signature
                or best_saved.get("hyperparameters") != hyperparameters
                or best_saved.get("epoch") != best_epoch):
            raise ValueError("best_model.pth tidak cocok dengan last_model.pth eksperimen ini.")
        del best_saved
    checkpoint_dir = run_dir / "checkpoints"
    metrics_dir = run_dir / "metrics"
    for directory in (checkpoint_dir, metrics_dir, run_dir / "predictions"):
        directory.mkdir(parents=True, exist_ok=True)
    best_path = checkpoint_dir / "best_model.pth"
    last_path = checkpoint_dir / "last_model.pth"
    save_json_atomic(metrics_dir / "configuration.json", metadata)
    seed_everything()
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    # num_workers=0 dan pin_memory=False aman untuk Windows/Jupyter; data tetap
    # dipindahkan ke GPU lewat .to(device), jadi CUDA tetap digunakan.
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=False,
        num_workers=0, pin_memory=False, collate_fn=collate_fn, generator=generator,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=False, collate_fn=collate_fn,
    )
    model = build_model(pretrained=bool(pretrained and resume is None)).to(device)
    optimizer = torch.optim.SGD(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    )
    if resumed_checkpoint is not None:
        model.load_state_dict(resumed_checkpoint["model_state_dict"], strict=True)
        optimizer.load_state_dict(resumed_checkpoint["optimizer_state_dict"])
        # Optimizer dibaca pada CPU; pindahkan momentumnya ke perangkat model.
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(device)
        generator.set_state(resumed_checkpoint["loader_rng_state"].cpu())
        restore_rng_state(resumed_checkpoint["rng_state"])
        del resumed_checkpoint
    print(f"Device: {device} | train={len(train_dataset)} | val={len(val_dataset)} | batch={batch_size}")
    print(f"Folder eksperimen: {run_dir}")
    try:
        for epoch_index in range(start_epoch, num_epochs):
            model.train()
            sums = {"training_loss": 0.0, "classification": 0.0, "bbox_regression": 0.0}
            seen_images, singleton_batches = 0, 0
            for images, targets in train_loader:
                images = [image.to(device) for image in images]
                targets = [{key: value.to(device) for key, value in target.items()} for target in targets]
                optimizer.zero_grad(set_to_none=True)
                with singleton_batch_norm(model, len(images)):
                    losses = model(images, targets)
                    if set(losses) != {"classification", "bbox_regression"}:
                        raise RuntimeError(f"Loss SSDLite tidak sesuai: {list(losses)}")
                    loss = sum(losses.values())
                    if not torch.isfinite(loss).item():
                        raise RuntimeError(f"Loss tidak finite pada epoch {epoch_index + 1}.")
                    loss.backward()
                    if any(not torch.isfinite(p.grad).all().item()
                           for p in model.parameters() if p.grad is not None):
                        raise RuntimeError("Gradien tidak finite; optimizer tidak dijalankan.")
                    optimizer.step()
                count = len(images)
                seen_images += count
                singleton_batches += int(count == 1)
                sums["training_loss"] += float(loss.detach()) * count
                for name in losses:
                    sums[name] += float(losses[name].detach()) * count
            if seen_images != len(train_dataset):
                raise RuntimeError("Ada gambar train yang belum diproses dalam epoch ini.")
            averages = {name: total / seen_images for name, total in sums.items()}
            validation = evaluate_model(
                model, val_loader, device, confidence_threshold=CONFIDENCE_THRESHOLD,
                iou_threshold=IOU_THRESHOLD,
            )
            validation_map = validation["map"]
            if validation_map is None or not math.isfinite(validation_map):
                raise RuntimeError("mAP validation tidak terdefinisi; periksa anotasi validation.")
            epoch = epoch_index + 1
            row = {
                "epoch": epoch, **averages, "learning_rate": optimizer.param_groups[0]["lr"],
                "val_map": validation_map, "val_map_50": validation["map_50"],
                "val_precision": validation["precision"], "val_recall": validation["recall"],
                "val_f1": validation["f1"], "train_images": seen_images,
                "singleton_batches": singleton_batches,
            }
            history.append(row)
            is_best = validation_map > best_map
            if is_best:
                best_map, best_epoch = validation_map, epoch
            checkpoint = {
                "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch, "training_loss": averages["training_loss"],
                "best_map": best_map, "best_epoch": best_epoch,
                "validation_metrics": validation, "history": history,
                "model_name": MODEL_NAME, "num_classes": NUM_CLASSES, "class_names": CLASS_NAMES,
                "config": metadata, "hyperparameters": hyperparameters, "dataset_signature": signature,
                "rng_state": capture_rng_state(), "loader_rng_state": generator.get_state(),
                "versions": {"torch": str(torch.__version__)},
            }
            if is_best:
                save_checkpoint_atomic(best_path, checkpoint)
                save_json_atomic(metrics_dir / "best_validation_metrics.json", validation)
            save_checkpoint_atomic(last_path, checkpoint)
            save_json_atomic(metrics_dir / "validation_metrics.json", validation)
            save_training_history(metrics_dir, history)
            print(
                f"Epoch {epoch}/{num_epochs} | loss={averages['training_loss']:.4f} "
                f"(classification={averages['classification']:.4f}, bbox={averages['bbox_regression']:.4f}) "
                f"| lr={row['learning_rate']:.6f} | val mAP={validation_map:.4f}"
            )
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            "Memori GPU tidak cukup. Restart Kernel, tutup aplikasi pemakai GPU, "
            "turunkan BATCH_SIZE menjadi 2, lalu mulai eksperimen baru. "
            "Checkpoint epoch yang sudah selesai tetap tersimpan; tidak ada perpindahan otomatis ke CPU."
        ) from error
    finally:
        del model, optimizer
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return {
        "best_checkpoint": str(best_path), "last_checkpoint": str(last_path),
        "run_dir": str(run_dir), "metrics_dir": str(metrics_dir),
        "best_epoch": best_epoch, "best_validation_map": best_map,
        "completed_epochs": num_epochs, "history": history,
    }


In [ ]:
training_result = train_experiment(
    datasets["train"], datasets["val"],
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    output_root=OUTPUT_ROOT, pretrained=USE_PRETRAINED,
    resume=RESUME_CHECKPOINT,
)
RUN_DIR = Path(training_result["run_dir"])
METRICS_DIR = Path(training_result["metrics_dir"])
save_json_atomic(METRICS_DIR / "dataset_audit.json", dataset_report)
print("Eksperimen:", RUN_DIR)
print("Epoch terbaik:", training_result["best_epoch"])
print("Best model:", training_result["best_checkpoint"])


## Tahap 8 — Membaca grafik training

Grafik menunjukkan loss training dan mAP validation sepanjang epoch. Loss turun
belum tentu berarti deteksi foto baru membaik; lihat validation dan variasi data.
Sel ini hanya membaca hasil run saat ini, tidak menambah epoch.

In [ ]:
history = training_result["history"]
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, label in [("training_loss", "Total"), ("classification", "Classification"),
                   ("bbox_regression", "BBox regression")]:
    axes[0].plot(epochs, [row[key] for row in history], label=label)
axes[0].set(title="Loss training", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(epochs, [row["val_map"] for row in history], label="mAP@0.5:0.95")
axes[1].plot(epochs, [row["val_map_50"] for row in history], label="mAP@0.5")
axes[1].axvline(training_result["best_epoch"], color="gray", linestyle="--", label="Epoch terbaik")
axes[1].set(title="Validation", xlabel="Epoch", ylabel="mAP", ylim=(0, 1))
axes[1].legend()
fig.tight_layout()
fig.savefig(METRICS_DIR / "training_curves.png", dpi=180)
plt.show()
plt.close(fig)


## Tahap 9 — Memuat checkpoint terbaik dan memilih confidence pada validation

Evaluasi memakai checkpoint **terbaik**, bukan otomatis epoch terakhir. Daftar
confidence di bawah dibandingkan menggunakan validation saja. Kode memilih F1
micro tertinggi (jumlah TP/FP/FN seluruh kelas); bila seri, pilih precision lalu
confidence lebih tinggi. Ini satu aturan pemilihan operasional, bukan jaminan
jumlah biji selalu benar. Periksa juga hasil visual dan performa setiap kelas.

mAP tidak berubah hanya karena confidence PR/F1 diganti, karena masukan mAP
tetap menggunakan skor rendah. Setelah threshold dipilih, bekukan sebelum test.

In [ ]:
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
best_checkpoint_path = Path(training_result["best_checkpoint"])
best_checkpoint = load_trusted_checkpoint(best_checkpoint_path)
best_model = build_model(pretrained=False).to(DEVICE)
best_model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
best_model.eval()
val_loader = DataLoader(datasets["val"], batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False, collate_fn=detection_collate)
print("Model terbaik dari epoch", best_checkpoint["epoch"])
del best_checkpoint


In [ ]:
CANDIDATE_CONFIDENCES = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
validation_results = []
for threshold in CANDIDATE_CONFIDENCES:
    values = evaluate_model(best_model, val_loader, DEVICE, threshold, IOU_THRESHOLD)
    validation_results.append(values)
    print(f"Confidence {threshold:.2f} | P={values['precision']:.4f} | "
          f"R={values['recall']:.4f} | F1={values['f1']:.4f}")
selected_validation = max(validation_results,
                          key=lambda row: (row["f1"], row["precision"], row["confidence_threshold"]))
SELECTED_CONFIDENCE = selected_validation["confidence_threshold"]
save_json_atomic(METRICS_DIR / "validation_confidence_sweep.json", validation_results)
save_json_atomic(METRICS_DIR / "selected_validation_metrics.json", selected_validation)
save_json_atomic(RUN_DIR / "inference_settings.json", {
    "checkpoint": str(best_checkpoint_path), "confidence_threshold": SELECTED_CONFIDENCE,
    "iou_threshold": IOU_THRESHOLD, "selection_split": "val", "rule": "max micro F1, precision, confidence",
})
print("Confidence terpilih:", SELECTED_CONFIDENCE)
show_metrics(selected_validation)


## Tahap 10 — Evaluasi final pada test

Aktifkan `JALANKAN_TEST_FINAL = True` ketika model dan threshold dari validation
sudah ditetapkan, lalu jalankan sel ini. Pengaturan awal `False` memungkinkan
eksperimen training/validation diulang tanpa memakai test untuk pengambilan
keputusan. Tidak perlu mengetik perintah terminal.

Test adalah laporan akhir pada gambar berlabel yang tidak dipakai training.
Jangan memilih epoch, confidence, atau mengubah hyperparameter berdasarkan
angka test. Jika kelas tidak ada pada test, AP kelas itu tidak dapat dinilai.

In [ ]:
JALANKAN_TEST_FINAL = False  # ubah True setelah pengaturan final dipilih dari validation
if JALANKAN_TEST_FINAL:
    test_loader = DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=False, collate_fn=detection_collate)
    test_metrics = evaluate_model(best_model, test_loader, DEVICE, SELECTED_CONFIDENCE, IOU_THRESHOLD)
    save_json_atomic(METRICS_DIR / "test_metrics.json", test_metrics)
    show_metrics(test_metrics)
    print("Laporan final:", METRICS_DIR / "test_metrics.json")
else:
    print("Test belum dijalankan. Aktifkan setelah pemilihan model/threshold final dari validation.")


## Tahap 11 — Menguji foto baru yang belum pernah dilihat model

Pilih gambar JPG/PNG/WEBP melalui tombol unggah di sel terakhir, atau isi
`PHOTO_PATH` jika ingin memakai file lokal. Tanpa anotasi, foto baru menghasilkan
deteksi dan hitungan, **bukan nilai akurasi/mAP**. Confidence awal mengikuti
pemilihan validation; slider hanya untuk eksplorasi visual foto baru dan tidak
mengubah laporan test yang sudah disimpan.

Hasil berupa gambar dengan bounding box, nama kelas, confidence, jumlah tiap
kelas, dan total. JPEG/PNG hasil serta JSON disimpan pada folder `predictions`
run saat ini. NMS bawaan model menangani box tumpang tindih; tidak ada pembatasan
jumlah biji berdasarkan perkiraan manual. Periksa hasil visual untuk biji yang
terlewat atau deteksi ganda.

In [ ]:
@torch.inference_mode()
def predict_new_image(source, confidence=None, display_result=True):
    confidence = SELECTED_CONFIDENCE if confidence is None else float(confidence)
    if not math.isfinite(confidence) or not 0 <= confidence <= 1:
        raise ValueError("Confidence harus berada antara 0 dan 1.")
    if isinstance(source, (bytes, bytearray, memoryview)):
        source = io.BytesIO(bytes(source))
    with Image.open(source) as opened:
        image = ImageOps.exif_transpose(opened).convert("RGB")
    best_model.eval()
    # score_thresh model tetap rendah. Filter confidence hanya setelah inferensi.
    prediction = best_model([F.to_tensor(image).to(DEVICE)])[0]
    keep = prediction["scores"] >= confidence
    prediction = {key: value[keep].detach().cpu() for key, value in prediction.items()}
    counts = {CLASS_NAMES[label]: int((prediction["labels"] == label).sum())
              for label in CLASS_NAMES if label > 0}
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default(size=max(12, round(max(image.size) / 90)))
    colors = {1: "#d62728", 2: "#e3a008", 3: "#238636"}
    width, height = image.size
    line_width = max(2, round(max(width, height) / 500))
    for box, label, score in zip(prediction["boxes"].tolist(), prediction["labels"].tolist(), prediction["scores"].tolist()):
        x1, y1, x2, y2 = box
        color = colors[label]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=line_width)
        text = f"{CLASS_NAMES[label]} {score:.2f}"
        bounds = draw.textbbox((0, 0), text, font=font)
        text_width, text_height = bounds[2] - bounds[0], bounds[3] - bounds[1]
        text_x = max(0, min(x1, width - text_width - 4))
        text_y = max(0, y1 - text_height - 6)
        draw.rectangle((text_x, text_y, min(width, text_x + text_width + 4),
                        min(height, text_y + text_height + 4)), fill=color)
        draw.text((text_x + 2, text_y + 2 - bounds[1]), text, font=font, fill="white")
    output_dir = RUN_DIR / "predictions"
    output_dir.mkdir(parents=True, exist_ok=True)
    name = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
    image_path = output_dir / f"{name}.jpg"
    json_path = output_dir / f"{name}.json"
    canvas.save(image_path, quality=95)
    result = {"boxes": prediction["boxes"].tolist(), "labels": prediction["labels"].tolist(),
              "scores": prediction["scores"].tolist(), "counts": counts,
              "total": sum(counts.values()), "confidence_threshold": confidence,
              "checkpoint": str(best_checkpoint_path), "image_path": str(image_path)}
    save_json_atomic(json_path, result)
    if display_result:
        display(canvas)
        for name, count in counts.items():
            print(f"{name}: {count}")
        print(f"Total: {result['total']} biji")
        print("Gambar:", image_path)
        print("JSON:", json_path)
    return result


In [ ]:
PHOTO_PATH = ""  # opsional: r"D:\foto\biji-baru.jpg"; kosong = gunakan tombol unggah
if PHOTO_PATH:
    photo_result = predict_new_image(Path(PHOTO_PATH))

photo_upload = widgets.FileUpload(accept="image/jpeg,image/png,image/webp", multiple=False)
confidence_slider = widgets.FloatSlider(value=SELECTED_CONFIDENCE, min=0.05, max=0.95,
                                         step=0.05, description="Confidence:",
                                         style={"description_width": "initial"})
predict_button = widgets.Button(description="Deteksi Foto Baru", button_style="success")
prediction_output = widgets.Output()

def run_photo_prediction(_):
    with prediction_output:
        clear_output(wait=True)
        predict_button.disabled = True
        try:
            if not photo_upload.value:
                print("Pilih satu foto dengan tombol unggah terlebih dahulu.")
                return
            item = photo_upload.value[0]
            predict_new_image(item["content"], confidence_slider.value)
        except Exception as error:
            print(f"Prediksi gagal: {error}")
        finally:
            predict_button.disabled = False

predict_button.on_click(run_photo_prediction)
display(widgets.VBox([widgets.HTML("<b>Pilih foto baru dari komputer Anda</b>"),
                      photo_upload, confidence_slider, predict_button, prediction_output]))


## Letak hasil dan cara melanjutkan training

```text
hasil_penelitian/eksperimen_<waktu>_<id>/
├── checkpoints/
│   ├── best_model.pth
│   └── last_model.pth
├── metrics/
│   ├── training_history.csv
│   ├── training_history.json
│   ├── training_curves.png
│   ├── dataset_audit.json
│   ├── best_validation_metrics.json
│   ├── validation_confidence_sweep.json
│   ├── selected_validation_metrics.json
│   └── test_metrics.json  # setelah sel test final diaktifkan
├── predictions/          # setelah foto baru diprediksi
└── inference_settings.json
```

Untuk **eksperimen baru**, biarkan `RESUME_CHECKPOINT=None`; setiap eksekusi sel
training membuat folder baru. Untuk **melanjutkan run notebook ini** yang terputus,
isi `RESUME_CHECKPOINT` dengan path `last_model.pth` dan atur `NUM_EPOCHS` sebagai
**total target epoch** yang lebih besar daripada epoch tersimpan. Misalnya model
berhenti pada epoch 40, target 100 berarti melanjutkan 41–100. Pertahankan dataset,
label, batch, learning rate, dan parameter lain; `best_model.pth` run tersebut
harus tetap berada di samping `last_model.pth`. Checkpoint dari program lain
tidak digunakan untuk resume notebook ini.

Setelah restart kernel, jalankan kembali sel dari atas untuk membentuk fungsi
dan dataset. Untuk demo prediksi tanpa training ulang, sel **muat-model-terbaik**
dapat diarahkan ke hasil run tersimpan: tambahkan tiga pengaturan berikut di awal
sel tersebut (sebelum `gc.collect()`), dan lewati sel training:

```python
RUN_DIR = Path(r"D:\...\hasil_penelitian\eksperimen_...")
METRICS_DIR = RUN_DIR / "metrics"
training_result = {"best_checkpoint": str(RUN_DIR / "checkpoints" / "best_model.pth")}
```

Lalu jalankan sel muat model, confidence validation (atau baca nilai final pada
`inference_settings.json`), fungsi prediksi, dan pemilih foto baru. Jangan menjalankan
sel training jika hanya ingin mempresentasikan model yang sudah selesai.

**Kendala umum:** `NameError` berarti ada sel sebelumnya yang belum dijalankan;
dataset tidak ditemukan berarti periksa `DATASET_DIR`; kelas tidak dikenali
berarti cocokkan `CATEGORY_ALIASES` dengan nama kategori JSON. Jika CUDA kehabisan
memori, Restart Kernel lalu pilih batch 2 untuk eksperimen baru. Program tidak
berpindah ke CPU diam-diam setelah kegagalan GPU. Menambah epoch saja tidak
menjamin seluruh biji terdeteksi; kualitas label, variasi foto, ukuran objek,
dan distribusi kelas juga menentukan hasil.